# Load the data

In [1]:
import os
import numpy as np
import random
import math
from collections import Counter

import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix
from sklearn.mixture import GaussianMixture
import umap

In [2]:
X_ls = []
y_ls = []
DATA_DIR = "../spectrogram_data/"
for x_file in sorted(os.listdir(DATA_DIR)):
    if "X" not in x_file:
        continue
    num_str = "".join([j for j in filter(str.isdigit, x_file)])
    X = np.load(DATA_DIR + x_file)
    y = np.load(f"{DATA_DIR}/sub-{num_str}_y.npy")
    assert X.shape[0] == y.shape[0]
    X_ls.append(X)
    y_ls.append(y)
    

# Set reference counts to 0 to try to reclaim space
X = np.concatenate(X_ls)
X_ls = None
y = np.concatenate(y_ls)
y_ls = None

X_mean = X.mean()
X_std = X.std()
X = (X - X_mean) / (X_std + 1e-8)
X_mean = None
X_std = None
print("Stacked shapes: ", X.shape, y.shape)

# Remove dementia patients
indicies = np.where(y!=2)[0]
y = y[indicies]
X = X[indicies]
Xshape = X.shape # When we want to reuse unflattened shape
print("Excluding dementia shapes: ", X.shape, y.shape)

# Flatten
X = X.reshape((X.shape[0], -1))
print("Flattened X shape: ", X.shape)
# Alzheimers = 1, Healthy = 0
print("Y counts: ", Counter(y))

# Split into Train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=67, shuffle=True
)
print(f"Xtr: {X_train.shape}, ytr: {y_train.shape}, Xte: {X_test.shape}, yte: {y_test.shape}")

Stacked shapes:  (69705, 19, 33, 7) (69705,)
Excluding dementia shapes:  (45637, 19, 33, 7) (45637,)
Flattened X shape:  (45637, 4389)
Y counts:  Counter({np.int64(1): 29081, np.int64(0): 16556})
Xtr: (36509, 4389), ytr: (36509,), Xte: (9128, 4389), yte: (9128,)


# Knn

In [ ]:
# Square root method says let k = sqrt(N)//2
math.sqrt(X_train.shape[0]) // 2

In [ ]:
# Generate Random Sampling from X and y
sampling_percentage = 0.5
indicies = random.sample(range(0, X_train.shape[0]), int(X_train.shape[0] * sampling_percentage))
print("Sampled label count: ", Counter(y_train[indicies]))

# Odd k so there is no tie
ks = [3, 5, 15, 31, 61, 91]
for k in ks
    neigh = KNeighborsClassifier(n_neighbors=k)
    neigh.fit(X_train[indicies], y_train[indicies])
    print(f"k = {k}")
    print("Train Score: ", neigh.score(X_train, y_train))
    print("Test Score: ", neigh.score(X_test, y_test))

# Visualization

In [ ]:
# TSNE
sampling_percentage = 0.4 # To reduce time and memory usage
indicies = random.sample(range(0, X_train.shape[0]), int(X_train.shape[0] * sampling_percentage))

perplexities = [5, 10, 15, 30, 50]
for perplexity in perplexities:
    X_embedded = TSNE(n_components=2, learning_rate='auto', init='random', perplexity=perplexity).fit_transform(X_train[indicies])
    alz_idx = np.where(y_train[indicies]==1)
    hea_idx = np.where(y_train[indicies]==0)
    plt.scatter(X_embedded[alz_idx, 0], X_embedded[alz_idx, 1], s=1, c="orange", label="Alzheimer")
    plt.scatter(X_embedded[hea_idx, 0], X_embedded[hea_idx, 1], s=1, c="blue", label="Healthy")
    plt.legend()
    plt.title(f"TSNE {sampling_percentage * 100}% of Training Data with P={perplexity}")
    plt.show()

In [ ]:
# UMAP
reducer = umap.UMAP()
embedding = reducer.fit_transform(X_train)
alz_idx = np.where(y_train==1)
hea_idx = np.where(y_train==0)
plt.scatter(embedding[alz_idx, 0], embedding[alz_idx, 1], s=1, c="orange", label="Alzheimer")
plt.scatter(embedding[hea_idx, 0], embedding[hea_idx, 1], s=1, c="blue", label="Healthy")
plt.xlabel("UMAP component 0")
plt.ylabel("UMAP component 1")
plt.title(f"UMAP of Training Data")
plt.show()

# Clustering

In [12]:
# Kmeans
from sklearn.cluster import MiniBatchKMeans
ks = [3, 4, 8, 12, 16, 20, 30, 40, 80, 120, 160, 200, 300, 400]

print("Percent Alz: " + str(Counter(y_train)[1] / len(y_train)))
print()
mappings = []
for k in ks:
    kmeans = MiniBatchKMeans(n_clusters=k, random_state=67, batch_size=64, max_iter=300).fit(X_train)
    labels = kmeans.predict(X_train)
    print(f"----------- k={k} -----------")
    correct = 0
    total = 0
    mapping = dict()
    for c in range(k):
        idx = np.where(labels == c)
        yhat = y_train[idx]
        l = len(yhat)
        total += l

        # print((yhat==1).sum(), l)
        alz_percent = (yhat==1).sum()/l
        # print(f"Cluster {c}: ", "{:.5f}".format(alz_percent), "with Alz. Size =", len(yhat))
        
        if alz_percent >= 0.5:
            num_correct = (yhat==1).sum()
            mapping[c] = 1
        else:
            mapping[c] = 0
            num_correct = l - (yhat==1).sum()

        correct += num_correct
    mappings.append(mapping)
    print(f"Accuracy for k={k}: ", correct / total)
        

        
        # print(f"Cluster {c}: ", "{:.5f}".format((yhat==1).sum()/l), "with Alz. Size =", len(yhat))
        # print("Healthy: ", (yhat==0).sum()/l)
        # print("Alz: ", (yhat==1).sum()/l)
    print()



Percent Alz: 0.6383905338409707

----------- k=3 -----------
Accuracy for k=3:  0.6383905338409707

----------- k=4 -----------
Accuracy for k=4:  0.6383905338409707

----------- k=8 -----------
Accuracy for k=8:  0.6534005313758251

----------- k=12 -----------
Accuracy for k=12:  0.6641923909173081

----------- k=16 -----------
Accuracy for k=16:  0.6561121915144211

----------- k=20 -----------
Accuracy for k=20:  0.6600290339368375

----------- k=30 -----------
Accuracy for k=30:  0.6819962201101099

----------- k=40 -----------
Accuracy for k=40:  0.6898025144484922

----------- k=80 -----------
Accuracy for k=80:  0.6876934454518064



/var/folders/d1/z8wvksb15lx10rvvj2835sh40000gn/T/ipykernel_21431/1031476470.py:22: RuntimeWarning: invalid value encountered in scalar divide
  alz_percent = (yhat==1).sum()/l


----------- k=120 -----------
Accuracy for k=120:  0.7392423785915802

----------- k=160 -----------
Accuracy for k=160:  0.6912268207839163



/var/folders/d1/z8wvksb15lx10rvvj2835sh40000gn/T/ipykernel_21431/1031476470.py:22: RuntimeWarning: invalid value encountered in scalar divide
  alz_percent = (yhat==1).sum()/l


----------- k=200 -----------
Accuracy for k=200:  0.719466432934345



/var/folders/d1/z8wvksb15lx10rvvj2835sh40000gn/T/ipykernel_21431/1031476470.py:22: RuntimeWarning: invalid value encountered in scalar divide
  alz_percent = (yhat==1).sum()/l


----------- k=300 -----------
Accuracy for k=300:  0.7031964721027691



/var/folders/d1/z8wvksb15lx10rvvj2835sh40000gn/T/ipykernel_21431/1031476470.py:22: RuntimeWarning: invalid value encountered in scalar divide
  alz_percent = (yhat==1).sum()/l


----------- k=400 -----------
Accuracy for k=400:  0.7006491550028761



/var/folders/d1/z8wvksb15lx10rvvj2835sh40000gn/T/ipykernel_21431/1031476470.py:22: RuntimeWarning: invalid value encountered in scalar divide
  alz_percent = (yhat==1).sum()/l


In [ ]:
# Gaussian Mixture Model
sampling_percentage = 0.2 # To reduce time and memory usage
indicies = random.sample(range(0, X_train.shape[0]), int(X_train.shape[0] * sampling_percentage))

gX = X_train[indicies]
gy = y_train[indicies]

ks = [2, 3, 5, 8]
for k in ks:
    labels = GaussianMixture(n_components=k, random_state=0).fit_predict(gX)
    k = max(labels)+1
    for c in range(k):
        idx = np.where(labels == c)
        yhat = gy[idx]
        l = len(yhat)
        print(f"Cluster {c}: ", "{:.5f}".format((yhat==1).sum()/l), "with Alz. Size =", len(yhat))
    print()

In [ ]:
from sklearn.cluster import AgglomerativeClustering

labels = AgglomerativeClustering().fit_predict(X_train)
k = max(labels)+1
for c in range(k):
    idx = np.where(labels == c)
    yhat = y_train[idx]
    l = len(yhat)
    print(f"Cluster {c}: ", "{:.5f}".format((yhat==1).sum()/l), "with Alz. Size =", len(yhat))
print()